# 🚕 Casablanca Synthetic Mobility Pipeline: NYC Projection

## Executive Summary
This notebook implements a high-fidelity spatial remapping pipeline. We take raw NYC Yellow Taxi mobility patterns and project them onto the urban landscape of Casablanca, Morocco. 

**Key Pipeline Stages:**
1. **MinIO Ingestion:** Direct streaming from S3 storage.
2. **Quality Enforcement:** Removal of anomalies (speed/duration).
3. **Spherical Trigonometry Projection:** Preserving distance statistics.
4. **Grid Zoning:** Discretizing the city into a 20x20 spatial matrix.
5. **Coastline Masking:** Enforcing physical urban boundaries.

## 1. Data Exploration & MinIO Ingestion

### The Methodology
We utilize `s3fs` to establish a stream to the MinIO cluster. Instead of loading the entire dataset into memory (which would exceed 1.5GB), we read the Parquet file metadata and extract a strategic sample of 10,000 rows. This ensures the pipeline is memory-efficient and scalable.

### The Analysis
In this step, we inspect the raw NYC columns: `trip_distance` (in miles) and the timestamps required for duration calculation.

In [1]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import folium
import s3fs
from folium.plugins import HeatMap

# Connection Configuration
fs = s3fs.S3FileSystem(key="minioadmin", secret="minioadmin", endpoint_url="http://minio:9000")
RAW_S3_PATH = "s3://taasim/raw/nyc-tlc/yellow_tripdata_2019-01.parquet"

def read_exploration_sample(rows=10000):
    with fs.open(RAW_S3_PATH, 'rb') as f:
        df = pq.ParquetFile(f).read_row_group(0, columns=[
            'tpep_pickup_datetime', 'tpep_dropoff_datetime', 
            'trip_distance', 'fare_amount', 'passenger_count',
            'PULocationID', 'DOLocationID'
        ]).to_pandas().head(rows)
        
    print("--- Raw NYC Data Exploration ---")
    print(f"Rows Loaded: {len(df)}")
    print(f"Avg NYC Trip Distance: {df['trip_distance'].mean():.2f} miles")
    return df

raw_data = read_exploration_sample()
raw_data.head()

--- Raw NYC Data Exploration ---
Rows Loaded: 10000
Avg NYC Trip Distance: 2.84 miles


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,passenger_count,PULocationID,DOLocationID
0,2019-01-01 00:46:40,2019-01-01 00:53:20,1.5,7.0,1.0,151,239
1,2019-01-01 00:59:47,2019-01-01 01:18:59,2.6,14.0,1.0,239,246
2,2018-12-21 13:48:30,2018-12-21 13:52:40,0.0,4.5,3.0,236,236
3,2018-11-28 15:52:25,2018-11-28 15:55:45,0.0,3.5,5.0,193,193
4,2018-11-28 15:56:57,2018-11-28 15:58:33,0.0,52.0,5.0,193,193


## 2. Unit Conversion & Quality Enforcement

### The Formula
We convert distances from the Imperial system (Miles) to the Metric system (Kilometers) using the standard factor:
$$Distance_{KM} = Distance_{Miles} \times 1.60934$$

### Logical Filtering
To ensure data integrity, we remove trips that are physically impossible (Speed > 100km/h) or logically inconsistent (Duration < 1min).

In [2]:
def clean_and_convert(df_in):
    # Create a proper copy to avoid SettingWithCopyWarning
    df = df_in.copy()
    
    # 1. Unit Conversion
    df['orig_dist_km'] = df['trip_distance'] * 1.60934
    
    # 2. Time-based Filtering
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    df['duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
    
    # 3. Anomaly Removal: Speed > 100km/h or Duration < 1min or > 3hrs
    initial_count = len(df)
    df = df[(df['duration_min'] >= 1) & (df['duration_min'] <= 180)].copy()
    df['avg_speed_kmh'] = df['orig_dist_km'] / (df['duration_min'] / 60)
    df = df[df['avg_speed_kmh'] <= 100].copy()
    
    # 4. Filter Fare/Distance outliers
    df = df[df['fare_amount'] > 2.5].copy()
    
    print(f"Filter Applied: {initial_count - len(df)} logical outliers removed.")
    return df

cleaned_df = clean_and_convert(raw_data)

Filter Applied: 127 logical outliers removed.


## 3. Spatial Remapping & Haversine Validation

### The Remapping Logic
We use a **Destination Point** spherical model. For every NYC trip, we pick a random valid start point in Casablanca and calculate a "Synthetic Destination" that exactly matches the original NYC trip distance.

### Haversine Validation
After projection, we apply a strict Haversine distance filter. We discard any trip that fails to meet the urban mobility constraints of Casablanca:
- **Minimum:** 0.5 KM
- **Maximum:** 20.0 KM

In [3]:
URBAN_BOUNDS = {"min_lon": -7.68, "max_lon": -7.58, "min_lat": 33.52, "max_lat": 33.60}

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def project_mobility(df_in):
    df = df_in.copy()
    n = len(df)
    R = 6371.0
    GRID_SIZE = 20
    
    # 1. Map IDs to Grid Indices
    # Deterministic mapping for start and target areas
    start_x = df['PULocationID'] % GRID_SIZE
    start_y = (df['PULocationID'] // GRID_SIZE).astype(int) % GRID_SIZE
    target_x = df['DOLocationID'] % GRID_SIZE
    target_y = (df['DOLocationID'] // GRID_SIZE).astype(int) % GRID_SIZE
    
    cell_width = (URBAN_BOUNDS['max_lon'] - URBAN_BOUNDS['min_lon']) / GRID_SIZE
    cell_height = (URBAN_BOUNDS['max_lat'] - URBAN_BOUNDS['min_lat']) / GRID_SIZE
    
    # 2. Derive Pickup point within the zone
    df['pickup_lon'] = URBAN_BOUNDS['min_lon'] + (start_x + np.random.uniform(0, 1, n)) * cell_width
    df['pickup_lat'] = URBAN_BOUNDS['min_lat'] + (start_y + np.random.uniform(0, 1, n)) * cell_height
    
    # 3. Calculate Bearing from zones
    # If Start == Target (intra-zone), fallback to random bearing
    bearing = np.where(
        df['PULocationID'] != df['DOLocationID'],
        np.arctan2(target_y - start_y, target_x - start_x),
        np.random.uniform(0, 2 * np.pi, n)
    )
    
    # 4. Project using Destination Point formula
    lat1_rad, lon1_rad = np.radians(df['pickup_lat']), np.radians(df['pickup_lon'])
    dist_r = df['orig_dist_km'] / R
    
    lat2_rad = np.arcsin(np.sin(lat1_rad) * np.cos(dist_r) + np.cos(lat1_rad) * np.sin(dist_r) * np.cos(bearing))
    lon2_rad = lon1_rad + np.arctan2(np.sin(bearing) * np.sin(dist_r) * np.cos(lat1_rad), 
                                    np.cos(dist_r) - np.sin(lat1_rad) * np.sin(lat2_rad))
    
    df['dropoff_lon'], df['dropoff_lat'] = np.degrees(lon2_rad), np.degrees(lat2_rad)
    
    # 5. Haversine Validation (0.5km - 20km)
    df['final_dist_km'] = haversine_km(df['pickup_lon'], df['pickup_lat'], df['dropoff_lon'], df['dropoff_lat'])
    df = df[(df['final_dist_km'] >= 0.5) & (df['final_dist_km'] <= 20)].copy()
    
    return df

projected_df = project_mobility(cleaned_df)

## 4. Grid Zoning & Coastline Masking

### Grid Discretization
To facilitate Demand Prediction, we partition the city into a **20x20 Grid Matrix**. Every coordinate is assigned a `zone_id` (e.g., `12_8`) calculated based on its offset from the bounding box origin.

### Coastline Boundary
We apply a diagonal linear boundary to ensure no trip landing or pickup occurs in the Atlantic Ocean.
$$Lat_{max} = (Lon \times 0.3) + 35.88$$

In [4]:
def apply_zoning_and_mask(df_in):
    df = df_in.copy()
    
    # 1. Grid Zoning Function
    def get_zone_id(lon, lat):
        x = int((lon - URBAN_BOUNDS['min_lon']) / (URBAN_BOUNDS['max_lon'] - URBAN_BOUNDS['min_lon']) * 20)
        y = int((lat - URBAN_BOUNDS['min_lat']) / (URBAN_BOUNDS['max_lat'] - URBAN_BOUNDS['min_lat']) * 20)
        return f"{x}_{y}"
    
    df['pickup_zone'] = df.apply(lambda r: get_zone_id(r['pickup_lon'], r['pickup_lat']), axis=1)
    df['dropoff_zone'] = df.apply(lambda r: get_zone_id(r['dropoff_lon'], r['dropoff_lat']), axis=1)
    
    # 2. Coastline Mask (Atlantic Ocean Removal)
    df = df[df['dropoff_lat'] < (df['dropoff_lon'] * 0.3 + 35.88)].copy()
    df = df[df['pickup_lat'] < (df['pickup_lon'] * 0.3 + 35.88)].copy()
    
    print(f"Final Dataset Ready. Logical Records: {len(df)}")
    return df

final_df = apply_zoning_and_mask(projected_df)

Final Dataset Ready. Logical Records: 7949


## 5. Visual Audit & Geographic Verification

We render the final dataset on an **OpenStreetMap** base. 
- **Green Circles:** Pickup Points
- **Red Circles:** Dropoff Points
- **Blue Vectors:** Predicted Trip Trajectories

In [5]:
m = folium.Map(location=[33.57, -7.63], zoom_start=13, tiles='OpenStreetMap')
sample = final_df.sample(min(100, len(final_df)))

for _, row in sample.iterrows():
    start, end = [row['pickup_lat'], row['pickup_lon']], [row['dropoff_lat'], row['dropoff_lon']]
    folium.PolyLine([start, end], color="blue", weight=2, opacity=0.6).add_to(m)
    folium.CircleMarker(start, radius=3, color='green', fill=True, popup=f"Zone: {row['pickup_zone']}").add_to(m)
    folium.CircleMarker(end, radius=3, color='red', fill=True, popup=f"Zone: {row['dropoff_zone']}").add_to(m)

m